In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../')

In [2]:
import numpy as np
import torch
import pandas as pd
from icecream import ic

from data_helpers.comp_equilibrium import dynamic_treatments
from data_helpers.data_loaders import TensorDataSet, TorchDataLoader

In [3]:
tds = TensorDataSet.deserialize('../data/preprocessed/torch/original_dataset')
splits_df = pd.read_feather('../data/preprocessed/train_test_split.ft')
for i in range(50):
    c_split_df = splits_df.query('sample_id == 3 and dataset_type=="test"')

tds_s = tds.subset(name='new_subset', treatment_idx=c_split_df['treatment_tnsr_idx'].tolist(),
                 game_idx=c_split_df['game_tnsr_idx'].tolist())

ic(tds_s.asks_tnsr[0, 0, 0, 3:10, :7])
ic(tds.asks_tnsr[c_split_df['treatment_tnsr_idx'].iloc[0], c_split_df['game_tnsr_idx'].iloc[0], 0, 3:10, :7])

tds_vars = vars(tds)

unique_treatment_idx = pd.unique(c_split_df['treatment_tnsr_idx'].values)
unique_game_idx = pd.unique(c_split_df['game_tnsr_idx'].values)

ic| tds_s.asks_tnsr[0, 0, 0, 3:10, :7]: tensor([[198.,   0.,   0.,   0.,   0.,   0.,   0.],
                                                [198., 150., 500.,   0.,   0.,   0.,   0.],
                                                [198., 150., 500.,   0.,   0.,   0.,   0.],
                                                [  0.,   0.,   0.,   0.,   0.,   0.,   0.],
                                                [198., 150., 500.,   0.,   0.,  88.,   0.],
                                                [198., 150., 500.,   0.,   0.,  88.,   0.],
                                                [198., 150., 400.,   0.,   0.,  88.,   0.]])
ic| tds.asks_tnsr[c_split_df['treatment_tnsr_idx'].iloc[0], c_split_df['game_tnsr_idx'].iloc[0], 0, 3:10, :7]: tensor([[198.,   0.,   0.,   0.,   0.,   0.,   0.],
                                                                                                                       [198., 150., 500.,   0.,   0.,   0.,   0.],
                             

In [4]:
asks_df = tds.reconstruct_player_dataframe(tds.asks_tnsr)

In [5]:
asks_df[asks_df['value']!=0].head(10)

,treatment,game,round,time,player_tnsr_id,value
150,BB,1,1,3,0,198.0
200,BB,1,1,4,0,198.0
201,BB,1,1,4,1,150.0
202,BB,1,1,4,2,500.0
250,BB,1,1,5,0,198.0
251,BB,1,1,5,1,150.0
252,BB,1,1,5,2,500.0
350,BB,1,1,7,0,198.0
351,BB,1,1,7,1,150.0
352,BB,1,1,7,2,500.0


In [6]:
import pandas as pd
df = pd.read_feather('../data/preprocessed/original_df.ft')

In [7]:
y0 = df.query('side=="Seller"')[['treatment', 'game', 'round', 'time', 'id', 'bid']]

In [8]:
def apply(row):
    kk = tuple(row[['treatment', 'game']].tolist())
    val = np.nan
    if kk in mapper:
        mapper_res = mapper[kk]
        k = str(row['player_tnsr_id'])
        val = mapper_res[k] if k in mapper_res else np.nan
    row['id'] = val
    return row
asks_df['id'] = np.nan


In [9]:
mapper = tds.user_index_df.set_index(['treatment', 'game']).to_dict(orient='index')
#asks_df['id'] = asks_df['player_tnsr_id'].to_frame().apply(lambda r: mapper[tuple(r.index)][r], axis=0)
y = asks_df.sample(10000).apply(apply,1).dropna()
y['id'] = y['id'].astype(int)
joined = pd.merge(y, y0, on=['treatment', 'game', 'round', 'time', 'id'])
assert (joined['bid'] == joined['value']).all()

In [10]:
# map to idx and columns are now keys of dict
# perform mapping
# inner join with fforward in mind.
# finish assert checks
# finish both data loaders
# add experiments and results